# 03: Model Training

## Objectives
- Set up data loaders and transforms
- Create CNN architecture for food classification
- Implement training pipeline with validation
- Use MLflow for experiment tracking
- Optimize hyperparameters with Optuna

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import mlflow
import mlflow.pytorch
import optuna
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

c:\Users\dell\Downloads\project ai\project ai\venv\lib\site-packages\pydantic\_internal\_config.py:386: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


Using device: cpu


## Configuration

In [2]:
DATA_DIR = Path('./data')
FOOD10_DIR = DATA_DIR / 'food10'
MODELS_DIR = Path('./models')
MLRUNS_DIR = Path('./mlruns')

# Create directories
MODELS_DIR.mkdir(exist_ok=True)
MLRUNS_DIR.mkdir(exist_ok=True)

SELECTED_CLASSES = [
    'pizza', 'sushi', 'hamburger', 'hot_dog', 'french_fries',
    'ice_cream', 'omelette', 'pancakes', 'ramen', 'steak'
]

CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(SELECTED_CLASSES)}
IDX_TO_CLASS = {idx: cls for cls, idx in CLASS_TO_IDX.items()}

# Training hyperparameters
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 0.001
IMG_SIZE = 224

print(f"Number of classes: {len(SELECTED_CLASSES)}")
print(f"Class mapping: {CLASS_TO_IDX}")

Number of classes: 10
Class mapping: {'pizza': 0, 'sushi': 1, 'hamburger': 2, 'hot_dog': 3, 'french_fries': 4, 'ice_cream': 5, 'omelette': 6, 'pancakes': 7, 'ramen': 8, 'steak': 9}


## Step 1: Custom Dataset Class

In [3]:
class FoodDataset(Dataset):
    def __init__(self, data_dir, split='train', transform=None):
        self.data_dir = Path(data_dir)
        self.split = split
        self.transform = transform
        
        # Load image paths and labels
        self.images = []
        self.labels = []
        
        split_dir = self.data_dir / split
        
        for class_name in SELECTED_CLASSES:
            class_dir = split_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob('*.jpg'):
                    self.images.append(img_path)
                    self.labels.append(CLASS_TO_IDX[class_name])
        
        print(f"Loaded {len(self.images)} {split} images")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, label

## Step 2: Data Transforms

In [4]:
def get_transforms(img_size=224):
    """Define data transformations"""
    
    # Training transforms with augmentation
    train_transform = transforms.Compose([
        transforms.Resize((img_size + 32, img_size + 32)),
        transforms.RandomCrop(img_size),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Validation/test transforms (no augmentation)
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_transform

train_transform, val_transform = get_transforms(IMG_SIZE)

## Step 3: Create Data Loaders

In [5]:
def create_data_loaders(batch_size=32):
    """Create training and validation data loaders"""
    
    # Create datasets
    train_dataset = FoodDataset(FOOD10_DIR, split='train', transform=train_transform)
    val_dataset = FoodDataset(FOOD10_DIR, split='test', transform=val_transform)
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=2,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    return train_loader, val_loader

train_loader, val_loader = create_data_loaders(BATCH_SIZE)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Loaded 6309 train images
Loaded 2104 test images
Training batches: 198
Validation batches: 66


## Step 4: Model Architecture

In [6]:
class FoodClassifier(nn.Module):
    def __init__(self, num_classes=10, pretrained=True):
        super(FoodClassifier, self).__init__()
        
        # Use ResNet18 as backbone
        self.backbone = models.resnet18(pretrained=pretrained)
        
        # Replace the final fully connected layer
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

def create_model(num_classes=10, pretrained=True):
    """Create and initialize the model"""
    model = FoodClassifier(num_classes=num_classes, pretrained=pretrained)
    model = model.to(device)
    return model

# Test model creation
model = create_model()
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

FoodClassifier(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, t

## Step 5: Training Functions

In [7]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()
        
        if batch_idx % 50 == 0:
            print(f'Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}')
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

def validate_epoch(model, val_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            
            running_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

## Step 6: Training Loop with MLflow

In [8]:
def train_model(model, train_loader, val_loader, num_epochs=50, lr=0.001, experiment_name="food_classifier"):
    """Train the model with MLflow tracking"""
    
    # Set up MLflow
    mlflow.set_experiment(experiment_name)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    
    best_val_acc = 0.0
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    
    with mlflow.start_run() as run:
        # Log parameters
        mlflow.log_param("model_type", "ResNet18_custom")
        mlflow.log_param("num_classes", len(SELECTED_CLASSES))
        mlflow.log_param("batch_size", BATCH_SIZE)
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("num_epochs", num_epochs)
        mlflow.log_param("img_size", IMG_SIZE)
        
        print(f"Starting training for {num_epochs} epochs...")
        
        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            print("-" * 50)
            
            # Train
            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
            
            # Validate
            val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
            
            # Update scheduler
            scheduler.step(val_loss)
            
            # Store metrics
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            train_accs.append(train_acc)
            val_accs.append(val_acc)
            
            # Log metrics
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("train_acc", train_acc, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            mlflow.log_metric("val_acc", val_acc, step=epoch)
            mlflow.log_metric("learning_rate", optimizer.param_groups[0]['lr'], step=epoch)
            
            print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
            print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
            print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
            
            # Save best model
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), MODELS_DIR / 'best_model.pth')
                print(f"New best model saved with val_acc: {val_acc:.2f}%")
                mlflow.log_metric("best_val_acc", best_val_acc)
        
        # Log model
        mlflow.pytorch.log_model(model, "model")
        
        # Save final model
        torch.save(model.state_dict(), MODELS_DIR / 'final_model.pth')
        
        print(f"\nTraining completed! Best validation accuracy: {best_val_acc:.2f}%")
    
    return {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs,
        'best_val_acc': best_val_acc
    }

## Step 7: Start Training

In [ ]:
# Create model
model = create_model(num_classes=len(SELECTED_CLASSES))

# Start training
training_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    experiment_name="food_classifier_v1"
)

Starting training for 50 epochs...

Epoch 1/50
--------------------------------------------------


## Step 8: Visualize Training History

In [ ]:
def plot_training_history(history):
    """Plot training history"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss plot
    axes[0].plot(history['train_losses'], label='Train Loss')
    axes[0].plot(history['val_losses'], label='Val Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Accuracy plot
    axes[1].plot(history['train_accs'], label='Train Acc')
    axes[1].plot(history['val_accs'], label='Val Acc')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nTraining Summary:")
    print(f"Best validation accuracy: {history['best_val_acc']:.2f}%")
    print(f"Final training accuracy: {history['train_accs'][-1]:.2f}%")
    print(f"Final validation accuracy: {history['val_accs'][-1]:.2f}%")

plot_training_history(training_history)

## Step 9: Hyperparameter Optimization with Optuna

In [ ]:
def objective(trial):
    """Optuna objective function for hyperparameter optimization"""
    
    # Define hyperparameters to optimize
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    
    # Create data loaders with new batch size
    train_loader_opt, val_loader_opt = create_data_loaders(batch_size=batch_size)
    
    # Create model
    model = create_model(num_classes=len(SELECTED_CLASSES))
    
    # Training setup
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    
    # Train for fewer epochs during optimization
    num_epochs_opt = 20
    best_val_acc = 0.0
    
    for epoch in range(num_epochs_opt):
        # Train
        train_loss, train_acc = train_epoch(model, train_loader_opt, criterion, optimizer, device)
        
        # Validate
        val_loss, val_acc = validate_epoch(model, val_loader_opt, criterion, device)
        
        # Update scheduler
        scheduler.step(val_loss)
        
        # Track best validation accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        
        # Prune if not improving
        trial.report(val_acc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    
    return best_val_acc

# Run hyperparameter optimization
def run_hyperparameter_optimization(n_trials=20):
    """Run hyperparameter optimization"""
    study = optuna.create_study(direction='maximize', study_name='food_classifier_optimization')
    study.optimize(objective, n_trials=n_trials)
    
    print("\nHyperparameter Optimization Results:")
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best validation accuracy: {study.best_trial.value:.2f}%")
    print("Best hyperparameters:")
    for key, value in study.best_trial.params.items():
        print(f"  {key}: {value}")
    
    return study

# Uncomment to run hyperparameter optimization (this will take a long time)
# study = run_hyperparameter_optimization(n_trials=10)

## Summary

In [ ]:
print("\n" + "=" * 50)
print("MODEL TRAINING SUMMARY")
print("=" * 50)
print(f"✅ Data loaders created")
print(f"✅ Model architecture defined (ResNet18-based)")
print(f"✅ Training pipeline implemented")
print(f"✅ MLflow experiment tracking set up")
print(f"✅ Model trained for {NUM_EPOCHS} epochs")
print(f"✅ Best model saved to {MODELS_DIR / 'best_model.pth'}")
print(f"✅ Training history visualized")
print(f"✅ Hyperparameter optimization framework ready")

print(f"\nModel Performance:")
print(f"- Best validation accuracy: {training_history['best_val_acc']:.2f}%")
print(f"- Model saved: {MODELS_DIR / 'best_model.pth'}")
print(f"- MLflow experiment: food_classifier_v1")

print("\nNext steps:")
print("1. Run 04_Model_Evaluation.ipynb for detailed evaluation")
print("2. Run 05_Inference.ipynb for testing on new images")
print("3. Consider hyperparameter optimization for better performance")